In [ ]:
# dependencies
# !pip install wikipedia-api pandas pyarrow tqdm

In [ ]:
# Imports and config
import pandas as pd
import numpy as np
import wikipediaapi
import json
import ast
import os
import re
import time
from tqdm import tqdm
 
# Config
PARQUET_PATH = "frames_test_preprocessed.parquet"   
OUTPUT_DIR   = "data"                               
CHUNK_SIZE   = 100   
CHUNK_OVERLAP = 0   
 
os.makedirs(OUTPUT_DIR, exist_ok=True)

Imports OK


/opt/anaconda3/lib/python3.13/site-packages/pandas/core/computation/expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


In [3]:
# Load FRAMES and extract unique Wikipedia URLs
df = pd.read_parquet(PARQUET_PATH)
print(f"Loaded {len(df)} questions")

# Parse it into actual Python lists.
def parse_wiki_links(raw):
    """Safely parse the wiki_links column into a list of URLs."""
    if isinstance(raw, list):
        return raw
    try:
        return ast.literal_eval(raw)
    except (ValueError, SyntaxError):
        # Try JSON parsing as fallback
        try:
            return json.loads(raw)
        except:
            return []
 
df["wiki_links_parsed"] = df["wiki_links"].apply(parse_wiki_links)
 
# Flatten all URLs and deduplicate
all_urls = []
for links in df["wiki_links_parsed"]:
    all_urls.extend(links)
 
unique_urls = list(set(all_urls))
print(f"Total URL references: {len(all_urls)}")
print(f"Unique articles to fetch: {len(unique_urls)}")
print(f"Example URL: {unique_urls[0]}")

Loaded 824 questions
Total URL references: 2636
Unique articles to fetch: 2507
Example URL: https://en.wikipedia.org/wiki/Pittsburgh


In [ ]:
# extract article title from Wikipedia URL
def url_to_title(url):
    """
    Extract the Wikipedia article title from a URL.
     'https://en.wikipedia.org/wiki/James_Buchanan' -> 'James Buchanan'
 
    Handles:
    - URL-encoded characters
    - Section anchors
    - Mobile URLs
    """
    from urllib.parse import unquote
 
    url = url.strip()
    # Remove section anchors
    url = url.split("#")[0]
    # Remove query parameters
    url = url.split("?")[0]
 
    # Extract the part after /wiki/
    if "/wiki/" in url:
        title = url.split("/wiki/")[-1]
    else:
        # Fallback: take the last part of the path
        title = url.rstrip("/").split("/")[-1]
 
    # Decode URL encoding and replace underscores with spaces
    title = unquote(title).replace("_", " ")
    return title
 
# test
test_urls = [
    "https://en.wikipedia.org/wiki/James_Buchanan",
    "https://en.wikipedia.org/wiki/Charlotte_Bront%C3%AB",
    "https://en.wikipedia.org/wiki/Blue_moon#Blue_moon_dates",
]
for u in test_urls:
    print(f"  {u} -> '{url_to_title(u)}'")


  https://en.wikipedia.org/wiki/James_Buchanan -> 'James Buchanan'
  https://en.wikipedia.org/wiki/Charlotte_Bront%C3%AB -> 'Charlotte Brontë'
  https://en.wikipedia.org/wiki/Blue_moon#Blue_moon_dates -> 'Blue moon'


In [ ]:
# Fetch all articles from Wikipedia API
 
CACHE_PATH = os.path.join(OUTPUT_DIR, "wiki_articles_raw.json")
 
# API etiquette
wiki = wikipediaapi.Wikipedia(
    user_agent="INFO371",
    language="en"
)
 
def fetch_article(title):
    """
    Fetch a single Wikipedia article's text.
    Returns (title, text) or (title, None) if not found.
    """
    page = wiki.page(title)
    if page.exists():
        return page.title, page.text
    return title, None

# Check results
if os.path.exists(CACHE_PATH):
    print(f"Loading cached articles from {CACHE_PATH}")
    with open(CACHE_PATH, "r", encoding="utf-8") as f:
        articles = json.load(f)
    print(f"Loaded {len(articles)} cached articles")
else:
    print(f"Fetching {len(unique_urls)} articles from Wikipedia API...")
 
    articles = {} 
    failed = []
 
    for url in tqdm(unique_urls, desc="Fetching"):
        title = url_to_title(url)
        try:
            resolved_title, text = fetch_article(title)
            if text:
                articles[url] = {
                    "title": resolved_title,
                    "text": text,
                    "status": "ok"
                }
            else:
                articles[url] = {
                    "title": title,
                    "text": None,
                    "status": "not_found"
                }
                failed.append((url, title))
        except Exception as e:
            articles[url] = {
                "title": title,
                "text": None,
                "status": f"error: {str(e)}"
            }
            failed.append((url, title))
 
        # basic api etiquette 
        time.sleep(0.1)
 
    # Save cache
    with open(CACHE_PATH, "w", encoding="utf-8") as f:
        json.dump(articles, f, ensure_ascii=False)
 
    print(f"\n Fetched {sum(1 for a in articles.values() if a['status'] == 'ok')} articles successfully")
    if failed:
        print(f"Failed to fetch {len(failed)} articles:")
        for url, title in failed[:10]:
            print(f"  - {title} ({url})")

Fetching 2507 articles from Wikipedia API...
This will take ~10-20 minutes. Progress below:



Fetching: 100%|██████████| 2507/2507 [39:51<00:00,  1.05it/s]   



Done! Fetched 2497 articles successfully
Failed to fetch 10 articles:
  - 2021 French Open – Men%27s singles (https://en.wikipedia.org/wiki/2021_French_Open_%E2%80%93_Men%2527s_singles)
  - Globe Life Field, (https://en.wikipedia.org/wiki/American_Family_Field, https://en.wikipedia.org/wiki/LoanDepot_Park, https://en.wikipedia.org/wiki/Globe_Life_Field, )
  - Noam Chomsky (https://en.wikipedia.org/wiki/Noam_Chomsky#In_academia)
  - Module:Location map/data/USA New York City (https://en.wikipedia.org/wiki/Module:Location_map/data/USA_New_York_City)
  - Pokémon (NOT REQUIRED, BUT HELPFUL) (https://en.wikipedia.org/wiki/Pok%C3%A9mon (NOT REQUIRED, BUT HELPFUL) )
  - ASFv (https://w.wiki/ASFv)
  - Category:Summer Olympics in London (https://en.wikipedia.org/wiki/Category:Summer_Olympics_in_London)
  - Jack Vance (tennis) (https://en.wikipedia.org/wiki/Jack_Vance_(tennis))
  - Nemanja Marković (https://en.wikipedia.org/wiki/Nemanja_Markovi%C4%87)
  - Alex Hernández (tennis), (https://en.wi

In [6]:
# Summary statistics on fetched articles
successful = {url: a for url, a in articles.items() if a["status"] == "ok"}
failed_articles = {url: a for url, a in articles.items() if a["status"] != "ok"}
 
print(f"Successfully fetched: {len(successful)} / {len(articles)}")
print(f"Failed:              {len(failed_articles)}")
 
# Article length distribution
lengths = [len(a["text"].split()) for a in successful.values()]
print(f"\nArticle word counts:")
print(f"  Min:    {min(lengths)}")
print(f"  Median: {int(np.median(lengths))}")
print(f"  Mean:   {int(np.mean(lengths))}")
print(f"  Max:    {max(lengths)}")
print(f"  Total words across all articles: {sum(lengths):,}")
 
if failed_articles:
    print(f"\nFailed articles (first 10):")
    for url, a in list(failed_articles.items())[:10]:
        print(f"  {a['title']} — {a['status']}")
 

Successfully fetched: 2497 / 2507
Failed:              10

Article word counts:
  Min:    13
  Median: 2153
  Mean:   3781
  Max:    21684
  Total words across all articles: 9,442,516

Failed articles (first 10):
  2021 French Open – Men%27s singles — not_found
  Globe Life Field, — not_found
  Noam Chomsky — error: https://en.wikipedia.org/w/api.php
  Module:Location map/data/USA New York City — not_found
  Pokémon (NOT REQUIRED, BUT HELPFUL) — not_found
  ASFv — not_found
  Category:Summer Olympics in London — not_found
  Jack Vance (tennis) — not_found
  Nemanja Marković — not_found
  Alex Hernández (tennis), — not_found


In [ ]:
# Chunk articles into ~100-token passages
def chunk_article(text, title, url, chunk_size=100):
    """
    Split article text into non-overlapping passages of ~chunk_size words.
 
    Design choices:
    - Word-level splitting (not character or sentence). DPR uses ~100 tokens,
      and words ≈ tokens for English (roughly 1 word = 1.3 BPE tokens).
    - Non-overlapping: standard for DPR. Overlap helps recall slightly but
      doubles corpus size and encoding time — not worth it for a course project.
    - We keep short final chunks (>20 words) rather than discarding them,
      since they might contain the answer to a FRAMES question.
 
    Returns a list of dicts, each with passage_id, title, url, text.
    """
    words = text.split()
    chunks = []
 
    for start in range(0, len(words), chunk_size):
        chunk_words = words[start : start + chunk_size]
 
        if len(chunk_words) < 20 and start > 0:
            # Append to previous chunk instead of creating a tiny one
            if chunks:
                chunks[-1]["text"] += " " + " ".join(chunk_words)
            continue
 
        chunks.append({
            "title": title,
            "url": url,
            "text": " ".join(chunk_words),
            "chunk_index": len(chunks),
        })
 
    return chunks
 
# Build the full passage corpus
all_passages = []
for url, article in successful.items():
    chunks = chunk_article(
        text=article["text"],
        title=article["title"],
        url=url,
        chunk_size=CHUNK_SIZE
    )
    all_passages.extend(chunks)
 
# Assign global passage IDs
for i, p in enumerate(all_passages):
    p["passage_id"] = i
 
passage_df = pd.DataFrame(all_passages)
 
print(f"Total passages in corpus: {len(passage_df):,}")
print(f"From {passage_df['url'].nunique()} unique articles")
print(f"\nPassage word count stats:")
passage_df["word_count"] = passage_df["text"].apply(lambda x: len(x.split()))
print(f"  Min:    {passage_df['word_count'].min()}")
print(f"  Median: {int(passage_df['word_count'].median())}")
print(f"  Mean:   {int(passage_df['word_count'].mean())}")
print(f"  Max:    {passage_df['word_count'].max()}")
print(f"\nSample passage (passage_id=0):")
print(f"  Title: {passage_df.iloc[0]['title']}")
print(f"  Text:  {passage_df.iloc[0]['text'][:300]}...")

Total passages in corpus: 95,202
From 2497 unique articles

Passage word count stats:
  Min:    13
  Median: 100
  Mean:   99
  Max:    119

Sample passage (passage_id=0):
  Title: Pittsburgh
  Text:  Pittsburgh ( PITS-burg) is a city in Allegheny County, Pennsylvania, United States, and its county seat. Located in southwestern Pennsylvania where the Allegheny and Monongahela Rivers meet to form the Ohio River, it had a population of 302,971 at the 2020 census, making it the second-most populous ...


In [8]:
# Build question-to-gold-passage mapping needed for Recall@k evaluation
def build_gold_mapping(frames_df, passage_df):
    """
    For each FRAMES question, find which passage_ids come from
    the question's gold Wikipedia articles.
 
    This mapping is essential for computing Recall@k:
    'Did the retriever find at least one gold passage in top-k?'
    """
    gold_map = []  # list of {question_idx, gold_passage_ids, gold_urls}
 
    for idx, row in frames_df.iterrows():
        gold_urls = row["wiki_links_parsed"]
 
        # Find all passages that belong to any of the gold articles
        gold_passage_ids = passage_df[
            passage_df["url"].isin(gold_urls)
        ]["passage_id"].tolist()
 
        gold_map.append({
            "question_idx": idx,
            "prompt": row["Prompt"],
            "answer": row["Answer"],
            "gold_urls": gold_urls,
            "n_gold_articles": len(gold_urls),
            "n_gold_passages": len(gold_passage_ids),
            "gold_passage_ids": gold_passage_ids,
        })
 
    return pd.DataFrame(gold_map)
 
gold_df = build_gold_mapping(df, passage_df)
 
print(f"Questions with at least 1 gold passage: {(gold_df['n_gold_passages'] > 0).sum()} / {len(gold_df)}")
print(f"Questions with 0 gold passages (fetch failures): {(gold_df['n_gold_passages'] == 0).sum()}")
print(f"\nGold passages per question:")
print(f"  Min:    {gold_df['n_gold_passages'].min()}")
print(f"  Median: {int(gold_df['n_gold_passages'].median())}")
print(f"  Mean:   {gold_df['n_gold_passages'].mean():.1f}")
print(f"  Max:    {gold_df['n_gold_passages'].max()}")

Questions with at least 1 gold passage: 824 / 824
Questions with 0 gold passages (fetch failures): 0

Gold passages per question:
  Min:    2
  Median: 105
  Mean:   126.4
  Max:    898


In [ ]:

# Passage corpus for DPR encoding and retrieval
passage_df.to_parquet(os.path.join(OUTPUT_DIR, "passage_corpus.parquet"), index=False)
passage_df.to_csv(os.path.join(OUTPUT_DIR, "passage_corpus.csv"), index=False)
 
# Gold mapping for Recall@k evaluation
gold_df.to_parquet(os.path.join(OUTPUT_DIR, "gold_mapping.parquet"), index=False)
 
# lightweight version, only passage_id and text
passage_df[["passage_id", "text"]].to_csv(
    os.path.join(OUTPUT_DIR, "passages_text_only.csv"), index=False
)
 
print(f"Saved to {OUTPUT_DIR}/:")
print(f"  passage_corpus.parquet  — full corpus ({len(passage_df):,} passages)")
print(f"  gold_mapping.parquet    — question-to-gold-passage map ({len(gold_df)} questions)")
 

Saved to data/:
  passage_corpus.parquet  — full corpus (95,202 passages)
  gold_mapping.parquet    — question-to-gold-passage map (824 questions)
  passages_text_only.csv  — lightweight text-only version
